# 02 - nnU-Net v2 baselines (3d_fullres and Residual Encoder M)

Runtime -> **T4 GPU**. Uses the *same* train/val split as the custom models (fold 0 of `splits_final.json`);
test cases are held out in `imagesTs`. Raw/preprocessed data live on local disk (rebuilt each session, a few minutes);
**results/checkpoints live on Drive**. `nnUNetTrainer_UMD` caps epochs and saves `checkpoint_latest` every 5 epochs
(stock nnU-Net: every 50), and re-running the train cell resumes with `--c`.

ResEnc **M** is used because it fits a 16 GB T4; ResEnc L/XL need 24 GB+.

In [ ]:
VARIANT = '3d_fullres'   # '3d_fullres' (nnU-Net default plans) | 'resenc' (nnUNetResEncUNetMPlans)
SMOKE = False             # True: 3-epoch timing run into a separate smoke/ folder. False: the real run
EPOCHS = 198             # set from the smoke-test timing; identical for both variants and across sessions

In [ ]:
# --- shared setup: Drive + code ---------------------------------------------
DRIVE = '/content/drive/MyDrive/figo'          # shared project folder (one Colab account)
REPO_URL = 'https://github.com/acm-research-f26/Radii.git'  # public; if made private see README (token in Colab Secrets)
REPO = '/content/figo-repo'

import os, subprocess
try:  # private repo: read-only token stored as GITHUB_TOKEN in Colab Secrets
    from google.colab import userdata
    REPO_URL = REPO_URL.replace('https://', f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}
!pip -q install -r requirements.txt
!pip -q install 'nnunetv2>=2.5'  # nnUNetTrainer_UMD matches the >=2.5 __init__ signature

/content/figo-repo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10

In [ ]:
!ls -la --time-style=full-iso {DRIVE}/nnUNet_results/Dataset501_UMD/nnUNetTrainer_UMD__nnUNetPlans__3d_fullres/fold_0

In [ ]:
from pathlib import Path
inp = Path('/kaggle/input')
roots = [p.parent for p in inp.rglob('UMD-001') if p.is_dir()]
print('dataset root(s):', roots)
root = roots[0]
for d in sorted(root.iterdir()):
    n = sum(1 for p in d.rglob('*') if p.is_file()) if d.is_dir() else 1
    kids = sorted(c.name for c in d.iterdir())[:6] if d.is_dir() else []
    print(f'{d.name}: {n} files | top-level: {kids}')
print('\nsplits.json:', [str(p) for p in root.rglob('splits.json')])
print('clean dirs:', [str(p) for p in root.rglob('clean') if p.is_dir()][:5])
print('cache dirs:', [str(p) for p in root.rglob('cache_0.9mm') if p.is_dir()][:5])
print('tar files:', [str(p) for p in root.rglob('*.tar')][:5])

dataset root(s): [PosixPath('/kaggle/input/datasets/sathya00/figo-dataset')]
UMD-001: 7445 files | top-level: ['UMD']
figo-20261021T165855Z-1-003: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-004: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-005: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-006: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-007: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-008: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-009: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-010: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-011: 112 files | top-level: ['figo']
figo-20261021T165855Z-1-012: 112 files | top-level: ['figo']
nnUNetPlans-014: 964 files | top-level: ['Dataset501_UMD']
nnUNetResEncUNetMPlans-013: 964 files | top-level: ['Dataset501_UMD']

splits.json: ['/kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-004/figo/data/splits.json']
clean dirs: ['/kaggle/input/da

In [ ]:
import os, shutil
from pathlib import Path

DRIVE = '/kaggle/working/figo'     # Kaggle has no Drive; this replaces the Colab path
DRY = False                       # first run: just look. Set to False to copy.
inp = Path('/kaggle/input')

splits = sorted(inp.rglob('splits.json'))
cleans = sorted(p for p in inp.rglob('clean') if p.is_dir())
print('splits.json:', [str(s) for s in splits])
print('clean folders:')
for c in cleans:
    n = sum(1 for _ in c.rglob('*.nii*'))
    print(f'  {c}  ({n} nifti files)')

if not DRY:
    assert splits, 'splits.json not found in any attached part (the part holding it may be missing)'
    os.makedirs(f'{DRIVE}/data/clean', exist_ok=True)
    shutil.copy(splits[0], f'{DRIVE}/data/splits.json')
    for c in cleans:
        shutil.copytree(c, f'{DRIVE}/data/clean', dirs_exist_ok=True)
    n = sum(1 for _ in Path(f'{DRIVE}/data/clean').rglob('*.nii*'))
    print('copied. nifti files in data/clean:', n, '(expect about 600: 300 images + 300 labels)')

splits.json: ['/kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-004/figo/data/splits.json']
clean folders:
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-003/figo/data/clean  (61 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-004/figo/data/clean  (59 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-005/figo/data/clean  (62 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-006/figo/data/clean  (59 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-007/figo/data/clean  (61 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-008/figo/data/clean  (59 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-009/figo/data/clean  (63 nifti files)
  /kaggle/input/datasets/sathya00/figo-dataset/figo-20261021T165855Z-1-010/figo/data/clean  (56 nifti files)
 

In [ ]:
import os, subprocess
os.makedirs('/content', exist_ok=True)          # the notebook hard-codes /content/... paths
DRIVE = '/kaggle/working/figo'

REPO_URL = 'https://github.com/acm-research-f26/Radii.git'
REPO = '/content/figo-repo'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}

# install requirements WITHOUT touching torch (Kaggle's GPU build is already there)
!grep -viE "^(torch|torchvision|torchaudio)" requirements.txt > /tmp/req.txt
!pip -q install -r /tmp/req.txt

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

/content/figo-repo
torch 2.11.0+cu128 | cuda: True


In [ ]:
!echo "--- figo ---"
!ls -la {DRIVE}
!echo "--- figo/data ---"
!ls -la {DRIVE}/data
!echo "--- anything named split* under figo ---"
!find {DRIVE} -maxdepth 4 -iname "*split*"

--- figo ---
total 12
drwxr-xr-x 3 root root 4096 Oct 22 00:42 .
drwxr-xr-x 4 root root 4096 Oct 22 00:42 ..
drwxr-xr-x 3 root root 4096 Oct 22 00:42 data
--- figo/data ---
total 56
drwxr-xr-x 3 root root  4096 Oct 22 00:42 .
drwxr-xr-x 3 root root  4096 Oct 22 00:42 ..
drwxrwxrwx 2 root root 36864 Oct 21 23:54 clean
-rw-rw-rw- 1 root root  6057 Oct 22 00:42 splits.json
--- anything named split* under figo ---
/kaggle/working/figo/data/splits.json


In [ ]:
import re, nnunetv2
from pathlib import Path

sig = re.compile(r'def __init__\(self,\s*\*args,\s*\*\*kwargs\)\s*:')
sup = re.compile(r'super\(\)\.__init__\(\s*\*args,\s*\*\*kwargs\s*\)')
NEW_SIG = "def __init__(self, plans, configuration, fold, dataset_json, device=None):"
NEW_SUP = ("import torch\n        super().__init__(plans, configuration, fold, dataset_json,\n"
           "                         device if device is not None else torch.device('cuda'))")

patched = []
for root in (Path('/content/figo-repo'), Path(nnunetv2.__file__).parent):
    for f in root.rglob('*.py'):
        txt = f.read_text()
        if 'nnUNetTrainer_UMD' in txt and sig.search(txt) and sup.search(txt):
            f.write_text(sup.sub(NEW_SUP, sig.sub(NEW_SIG, txt, count=1), count=1))
            patched.append(str(f))
print('patched:', patched)

patched: []


In [ ]:
# Preprocessing is CPU work: run up to this cell once on a CPU runtime; it is cached on Drive
# and later GPU sessions just unpack it (saves GPU units every session).
import subprocess
pre = f"/content/nnUNet_preprocessed/Dataset501_UMD/{PLANS}_3d_fullres"
tar = f"{DRIVE}/nnunet_cache/{PLANS}.tar"
if not os.path.isdir(pre):
    if os.path.exists(tar):
        !tar -xf {tar} -C /content/nnUNet_preprocessed
    else:
        subprocess.run(['nnUNetv2_plan_and_preprocess', '-d', '501', '-c', '3d_fullres',
                        '--verify_dataset_integrity', *PLANNER], check=True)
        os.makedirs(f'{DRIVE}/nnunet_cache', exist_ok=True)

import os, subprocess
VAL_PRED = f"{model_dir}/fold_0/validation"
print(len([f for f in os.listdir(VAL_PRED) if f.endswith('.nii.gz')]), "val predictions")

subprocess.run(['python', '-m', 'figo.evaluate',
                '--pred-dir', VAL_PRED, '--data-dir', f'{DRIVE}/data/clean',
                '--splits', f'{DRIVE}/data/splits.json',
                '--subset', 'val', '--out', f'{RES}/{NAME}_val_seg.csv'], check=True)

subprocess.run(['python', '-m', 'figo.figo_eval', 'agreement',
                '--ref-dir', f'{DRIVE}/data/clean', '--pred-dir', VAL_PRED,
                '--splits', f'{DRIVE}/data/splits.json',
                '--subset', 'val', '--out', f'{RES}/{NAME}_val_figo_agreement.csv'], check=True)

NameError: name 'PLANS' is not defined

In [ ]:
fold_dir = f"{os.environ['nnUNet_results']}/Dataset501_UMD/nnUNetTrainer_UMD__{PLANS}__3d_fullres/fold_0"
resume = ['--c'] if os.path.exists(f'{fold_dir}/checkpoint_latest.pth') else []
print('resuming' if resume else 'starting fresh', fold_dir)
!nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainer_UMD -p {PLANS} {' '.join(resume)}

In [ ]:
# timing: nnU-Net logs one "Epoch time" line per epoch (the first is slower: warm-up)
!grep -h "Epoch time" {fold_dir}/training_log_*.txt
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!nproc

## Test-set inference and evaluation (after training)

In [ ]:
NAME = f'nnunet_{VARIANT}'
PRED = f'{DRIVE}/preds/{NAME}'
RES = f'{DRIVE}/results'
!nnUNetv2_predict -i $nnUNet_raw/Dataset501_UMD/imagesTs -o {PRED} -d 501 -c 3d_fullres -f 0 \
    -tr nnUNetTrainer_UMD -p {PLANS} -chk checkpoint_best.pth
!python -m figo.evaluate --pred-dir {PRED} --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --subset val --out {RES}/{NAME}_seg.csv
!python -m figo.figo_eval agreement --ref-dir {DRIVE}/data/clean --pred-dir {PRED} \
    --splits {DRIVE}/data/splits.json --subset val --out {RES}/{NAME}_figo_agreement.csv